# Train a PointNet on plausibilitree-DL features

Bare-bones single-head PointNet regression (PyTorch) trained on the
feature/label pairs produced by `prepare.py`:

- `features/<plot_uid>.npz` — `points` (N, 3) float32 (+ extra dims, meta)
- `labels/<plot_uid>.json` — `{"plot_uid": ..., "labels": {NUM_TREES: ..., VOL_TREES: ..., ...}}`

One plot = one sample. Pick the device (CPU / Apple GPU via MPS / CUDA),
the feature and label directories, and the label to predict below.

Requires: `torch`, `numpy` (and `matplotlib` only for the optional plot).
Install e.g. with `pip install torch numpy matplotlib jupyter` in your
training environment, then `jupyter lab` and open this notebook.

In [ ]:
import glob
import json
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

# ---- device: CPU / Apple integrated GPU (MPS) / CUDA cores ---------------
DEVICE_CHOICE = "auto"  # "auto" | "cpu" | "mps" | "cuda"


def pick_device(choice: str) -> torch.device:
    if choice in ("cuda", "auto") and torch.cuda.is_available():
        return torch.device("cuda")
    if choice in ("mps", "auto") and torch.backends.mps.is_available():
        return torch.device("mps")  # Apple integrated GPU
    if choice != "auto":
        print(f"warning: {choice!r} not available, falling back to CPU")
    return torch.device("cpu")


device = pick_device(DEVICE_CHOICE)
print("device:", device)

In [ ]:
# ---- user config -----------------------------------------------------------
FEATURES_DIR = "../dl_dataset/features"  # prepare.py -o <out>/features
LABELS_DIR = "../dl_dataset/labels"      # prepare.py -o <out>/labels
LABEL_KEY = "VOL_TREES"                  # which label to train on (single head)

N_POINTS = 1024     # each cloud is padded/tiled or truncated to this many points
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-3
VAL_FRAC = 0.2
SEED = 42
STANDARDIZE_LABELS = True  # z-score the label using train-set stats

# labels available in the label files (from the first one found):
_first = sorted(glob.glob(os.path.join(LABELS_DIR, "*.json")))
if _first:
    with open(_first[0]) as fh:
        print("available labels:", sorted(json.load(fh)["labels"].keys()))
else:
    raise FileNotFoundError(f"no label files in {LABELS_DIR!r} - run prepare.py first")

In [ ]:
class PlotDataset(Dataset):
    """One plot per sample: points (3, N_POINTS) from the feature .npz and the
    scalar label LABEL_KEY from the matching label .json (shared plot name)."""

    def __init__(self, features_dir, labels_dir, label_key, n_points):
        self.n_points = n_points
        self.label_key = label_key
        self.samples = []
        for feat in sorted(glob.glob(os.path.join(features_dir, "*.npz"))):
            name = os.path.splitext(os.path.basename(feat))[0]
            lab = os.path.join(labels_dir, name + ".json")
            if os.path.exists(lab):
                self.samples.append((feat, lab))
        if not self.samples:
            raise FileNotFoundError(
                f"no matching feature/label pairs in {features_dir!r} + {labels_dir!r}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        feat, lab = self.samples[i]
        with np.load(feat) as z:
            pts = z["points"].astype(np.float32)  # (N, 3)
        with open(lab) as fh:
            y = float(json.load(fh)["labels"][self.label_key])
        # fix the point count: tile if short, truncate if long
        n = len(pts)
        if n >= self.n_points:
            pts = pts[: self.n_points]
        else:
            reps = int(np.ceil(self.n_points / n))
            pts = np.tile(pts, (reps, 1))[: self.n_points]
        return torch.from_numpy(pts.T.copy()), torch.tensor(y, dtype=torch.float32)


dataset = PlotDataset(FEATURES_DIR, LABELS_DIR, LABEL_KEY, N_POINTS)
print(f"{len(dataset)} plot samples, label = {LABEL_KEY}")

In [ ]:
class PointNetReg(nn.Module):
    """Bare-bones PointNet regression head: shared point MLP -> global max
    pool -> FC head -> 1 scalar. (No input/feature T-nets.)"""

    def __init__(self):
        super().__init__()
        self.point_mlp = nn.Sequential(
            nn.Conv1d(3, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, x):  # x: (B, 3, N)
        x = self.point_mlp(x)
        x = torch.max(x, dim=2).values  # (B, 1024)
        return self.head(x).squeeze(1)  # (B,)


model = PointNetReg().to(device)
print(model)

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

n_val = max(1, int(round(len(dataset) * VAL_FRAC))) if len(dataset) > 1 else 0
n_train = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_train, n_val],
                                  generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE) if n_val else None

# optional label standardization from train-set stats
if STANDARDIZE_LABELS:
    ys = torch.stack([dataset[i][1] for i in train_set.indices])
    y_mean, y_std = ys.mean().item(), ys.std().item() or 1.0
else:
    y_mean, y_std = 0.0, 1.0
print(f"label stats (train): mean={y_mean:.6g} std={y_std:.6g}")

criterion = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), (y.to(device) - y_mean) / y_std
        opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * len(x)
    train_loss /= len(train_set)
    msg = f"epoch {epoch:3d}/{EPOCHS}  train MSE {train_loss:.6f}"
    if val_loader is not None:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), (y.to(device) - y_mean) / y_std
                val_loss += criterion(model(x), y).item() * len(x)
        msg += f"  val MSE {val_loss / len(val_set):.6f}"
    print(msg)

In [ ]:
# ---- evaluate on the validation split, in original label units ------------
if val_loader is not None:
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in val_loader:
            p = model(x.to(device)).cpu() * y_std + y_mean
            preds.append(p.numpy())
            trues.append(y.numpy() * y_std + y_mean)
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    rmse = float(np.sqrt(np.mean((preds - trues) ** 2)))
    mae = float(np.mean(np.abs(preds - trues)))
    print(f"val RMSE {rmse:.6g}   MAE {mae:.6g}   ({LABEL_KEY}, n={len(trues)})")

    try:
        import matplotlib.pyplot as plt
        plt.scatter(trues, preds, s=12)
        lims = [min(trues.min(), preds.min()), max(trues.max(), preds.max())]
        plt.plot(lims, lims, "r--")
        plt.xlabel(f"true {LABEL_KEY}")
        plt.ylabel(f"predicted {LABEL_KEY}")
        plt.show()
    except ImportError:
        print("matplotlib not installed - skipping plot")
else:
    print("only one sample - no validation split")